# 08 - Zero-Shot Classification with Ollama (llama3.1:8b)

This notebook classifies tweets using **Ollama** running locally — no API key, no internet required.

### Datasets
- **Manchester** - reliable vs misinformation
- **Monkeypox** - reliable vs misinformation
- **PHEME** - not_rumour vs rumour

### Approach
- Zero-Shot: single prompt per tweet, no in-context examples
- Chain-of-Thought (CoT): model reasons before classifying
- Dataset-specific prompt templates (Promptbook)
- Structured JSON output parsing with fallback
- Evaluation: same metrics as RoBERTa (F1, Precision, Recall, Accuracy, CM)
- Results saved to `results/predictions/` for comparison in notebook 09

### Requirements
- Ollama installed: https://ollama.com
- Model pulled: `ollama pull llama3.1`

## 0. Dataset Selection

In [ ]:
# ============================================================
# DATASET SELECTION — change this to switch datasets
# Options: 'manchester', 'monkeypox', 'pheme'
# ============================================================
DATASET = 'manchester'

DATASET_CONFIG = {
    'manchester': {
        'test':      '../../data/gold_standard/manchester_test.csv',
        'text_col':  'cleaned_tweet',
        'label_col': 'label',
        'label_map': {'reliable': 0, 'misinformation': 1},
        'label_names': ['reliable', 'misinformation'],
        'pos_label': 'misinformation',
        'topic': 'the 2017 Manchester Arena bombing',
        'class_a': 'reliable',
        'class_b': 'misinformation',
        'class_a_desc': 'factually accurate, verified, or plausible news about the event',
        'class_b_desc': 'false, unverified, or misleading claims — rumours, conspiracy theories, or fabricated stories',
    },
    'monkeypox': {
        'test':      '../../data/gold_standard/monkeypox_test.csv',
        'text_col':  'cleaned_tweet',
        'label_col': 'label',
        'label_map': {'reliable': 0, 'misinformation': 1},
        'label_names': ['reliable', 'misinformation'],
        'pos_label': 'misinformation',
        'topic': 'the 2022 Monkeypox (Mpox) outbreak',
        'class_a': 'reliable',
        'class_b': 'misinformation',
        'class_a_desc': 'factually accurate health information about Monkeypox symptoms, transmission, or treatment',
        'class_b_desc': 'false health claims, conspiracy theories, or misleading information about Monkeypox',
    },
    'pheme': {
        'test':      '../../data/gold_standard/pheme_test.csv',
        'text_col':  'cleaned_tweet',
        'label_col': 'label',
        'label_map': {'not_rumour': 0, 'rumour': 1},
        'label_names': ['not_rumour', 'rumour'],
        'pos_label': 'rumour',
        'topic': 'breaking news events (Charlie Hebdo attack 2015, Ferguson unrest 2014)',
        'class_a': 'not_rumour',
        'class_b': 'rumour',
        'class_a_desc': 'verified news, factual reporting, or confirmed information about the events',
        'class_b_desc': 'unverified claims, speculation, or information that has not been confirmed by credible sources',
    },
}

CFG = DATASET_CONFIG[DATASET]
print(f"Dataset: {DATASET.upper()}")
print(f"Topic: {CFG['topic']}")
print(f"Classes: {CFG['label_names']}")

## 1. Imports & Setup

In [ ]:
import os
import re
import time
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
import requests

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Paths
RESULTS_DIR = Path('../../results')
PREDS_DIR   = RESULTS_DIR / 'predictions'
FIGS_DIR    = RESULTS_DIR / 'figures'
for d in [PREDS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Imports ready.")

## 2. Ollama Setup

> Make sure Ollama is running: open a terminal and run `ollama serve`
> Then pull the model once: `ollama pull llama3.1`

In [ ]:
OLLAMA_MODEL   = 'llama3.1'       # Model name as shown in `ollama list`
OLLAMA_URL     = 'http://localhost:11434/api/generate'
OLLAMA_TIMEOUT = 120              # seconds per request

def check_ollama():
    """Verify Ollama is running and model is available."""
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=5)
        models = [m['name'] for m in r.json().get('models', [])]
        print(f"Ollama running. Available models: {models}")
        # Check if our model is present (name may include tag like llama3.1:latest)
        if not any(OLLAMA_MODEL in m for m in models):
            print(f"\nWARNING: '{OLLAMA_MODEL}' not found!")
            print(f"Run in terminal:  ollama pull {OLLAMA_MODEL}")
        else:
            print(f"\nModel '{OLLAMA_MODEL}' is ready.")
    except requests.exceptions.ConnectionError:
        print("ERROR: Ollama is not running!")
        print("Start it with:  ollama serve")

check_ollama()

## 3. Load Test Data

In [ ]:
df_test = pd.read_csv(CFG['test'])
df_test.dropna(subset=[CFG['text_col'], CFG['label_col']], inplace=True)
df_test[CFG['text_col']] = df_test[CFG['text_col']].astype(str)

LABEL_MAP = CFG['label_map']
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}

print(f"Test set: {len(df_test):,} samples")
print(f"\nLabel distribution:")
print(df_test[CFG['label_col']].value_counts())

## 4. Promptbook — Dataset-Specific Prompts

Each dataset gets a tailored system prompt + Chain-of-Thought instruction.

In [ ]:
def build_prompt(tweet_text: str, cfg: dict) -> str:
    """
    Build a Chain-of-Thought zero-shot prompt for tweet classification.
    
    The model must:
    1. Reason step-by-step about the tweet
    2. Output a structured JSON response with label + reasoning
    """
    prompt = f"""You are an expert fact-checker and misinformation analyst specializing in social media content.

Your task: Classify the following tweet about {cfg['topic']}.

CLASSES:
- "{cfg['class_a']}": {cfg['class_a_desc']}
- "{cfg['class_b']}": {cfg['class_b_desc']}

TWEET:
\"\"\"{tweet_text}\"\"\"

INSTRUCTIONS:
Think step-by-step before classifying. Consider:
1. What specific claim does the tweet make?
2. Does it present verifiable facts, or unverified/emotional claims?
3. Are there signals of misinformation: conspiracy language, extreme emotion, lack of sources, implausible claims?
4. What is your final classification?

Respond in this exact JSON format (no extra text before or after):
{{
  "reasoning": "<your step-by-step reasoning in 2-4 sentences>",
  "label": "{cfg['class_a']}" or "{cfg['class_b']}",
  "confidence": <float between 0.0 and 1.0>
}}"""
    return prompt


# Preview the prompt
sample_tweet = df_test[CFG['text_col']].iloc[0]
sample_prompt = build_prompt(sample_tweet, CFG)
print("=== PROMPT PREVIEW ===")
print(sample_prompt)
print("\n=== TWEET ===")
print(sample_tweet[:200])

## 5. Ollama Inference with Retry Logic

In [ ]:
def call_ollama(prompt: str, max_retries: int = 3) -> str:
    """
    Call Ollama REST API and return raw response text.
    Retries on connection errors with exponential backoff.
    Returns empty string on total failure (never raises).
    """
    payload = {
        'model':  OLLAMA_MODEL,
        'prompt': prompt,
        'stream': False,
        'options': {
            'temperature': 0.0,   # Deterministic
            'top_k': 1,
            'top_p': 1.0,
            'num_predict': 300,   # Max tokens in response
        }
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=OLLAMA_TIMEOUT)
            response.raise_for_status()
            raw = response.json().get('response', '')
            # Explicit null/empty guard — log and return empty string
            if raw is None:
                print(f"  [Warning] Model returned None on attempt {attempt+1}")
                raw = ''
            return raw
        except requests.exceptions.Timeout:
            print(f"  [Timeout] attempt {attempt+1}/{max_retries}")
            if attempt < max_retries - 1:
                time.sleep(5 * (attempt + 1))
        except requests.exceptions.RequestException as e:
            print(f"  [Error] {e}")
            if attempt < max_retries - 1:
                time.sleep(3)

    return ''


def parse_response(response_text: str, cfg: dict) -> dict:
    """
    Parse JSON response from model. Falls back to keyword matching.

    Returns:
        label        : predicted label string or None
        confidence   : float 0-1
        reasoning    : string
        parse_error  : bool (True when JSON parse failed)
        parse_method : 'json' | 'keyword_fallback' | 'empty' | 'default'
    """
    # --- Null / empty guard ---------------------------------------------------
    if not response_text or not response_text.strip():
        majority = cfg['label_names'][0]   # filled later; mark as 'empty' for now
        print(f"  [Warning] Empty/None response — assigning majority class placeholder")
        return {
            'label':        None,      # will be filled with majority class downstream
            'confidence':   0.5,
            'reasoning':    'Empty or None response from model',
            'parse_error':  True,
            'parse_method': 'empty',
        }

    # --- Try JSON parse -------------------------------------------------------
    try:
        clean = re.sub(r'```json\s*|```\s*', '', response_text).strip()
        match = re.search(r'\{.*\}', clean, re.DOTALL)
        if match:
            data = json.loads(match.group())
            label = str(data.get('label', '')).strip().lower()
            if label in cfg['label_map']:
                return {
                    'label':        label,
                    'confidence':   float(data.get('confidence', 0.5)),
                    'reasoning':    str(data.get('reasoning', '')),
                    'parse_error':  False,
                    'parse_method': 'json',
                }
    except (json.JSONDecodeError, ValueError, TypeError):
        pass

    # --- Fallback: find label keyword in raw text ----------------------------
    text_lower = response_text.lower()
    for label_name in sorted(cfg['label_names'], key=len, reverse=True):
        if label_name in text_lower:
            return {
                'label':        label_name,
                'confidence':   0.5,
                'reasoning':    response_text[:300],
                'parse_error':  True,
                'parse_method': 'keyword_fallback',
            }

    return {
        'label':        None,
        'confidence':   0.0,
        'reasoning':    response_text[:300],
        'parse_error':  True,
        'parse_method': 'default',
    }


def classify_tweet(tweet: str, cfg: dict) -> dict:
    """Classify a single tweet using Ollama."""
    prompt = build_prompt(tweet, cfg)
    raw    = call_ollama(prompt)
    return parse_response(raw, cfg)


print("Inference functions defined.")
print(f"Model: {OLLAMA_MODEL} | Timeout: {OLLAMA_TIMEOUT}s")

## 6. Run Classification on Test Set

> Ollama runs locally — no rate limits. For ~375 tweets with llama3.1:8b expect ~10-20 minutes depending on GPU/CPU.

In [ ]:
CHECKPOINT_EVERY = 50   # save partial results every N tweets

raw_path        = PREDS_DIR / f'{DATASET}_ollama_raw.csv'
checkpoint_path = PREDS_DIR / f'{DATASET}_ollama_checkpoint.csv'

# ── Resume from checkpoint if it exists ──────────────────────────────────────
done_indices = set()
results      = []

if checkpoint_path.exists():
    df_ckpt = pd.read_csv(checkpoint_path)
    done_indices = set(df_ckpt['index'].tolist())
    results      = df_ckpt.to_dict('records')
    print(f"Checkpoint found: {len(done_indices):,} tweets already processed — resuming.")
else:
    print("No checkpoint found — starting fresh.")

# ── Filter to rows not yet processed ─────────────────────────────────────────
df_todo = df_test[~df_test.index.isin(done_indices)]
total   = len(df_test)

print(f"\nClassifying {len(df_todo):,} remaining tweets with Ollama ({OLLAMA_MODEL})...")
print(f"(Total test set: {total:,} | Already done: {len(done_indices):,})\n")

start_time     = time.time()
request_times  = []   # per-request timing for ETA

for n, (i, row) in enumerate(tqdm(df_todo.iterrows(), total=len(df_todo), desc="Classifying"), start=1):
    tweet      = row[CFG['text_col']]
    true_label = row[CFG['label_col']]

    t0     = time.time()
    result = classify_tweet(tweet, CFG)
    t1     = time.time()
    request_times.append(t1 - t0)

    results.append({
        'index':        i,
        'text':         tweet,
        'true_label':   true_label,
        'pred_label':   result['label'],
        'confidence':   result['confidence'],
        'reasoning':    result['reasoning'],
        'parse_error':  result['parse_error'],
        'parse_method': result['parse_method'],
    })

    # ── Progress + ETA ───────────────────────────────────────────────────────
    processed_total = len(done_indices) + n
    pct_done        = processed_total / total * 100
    avg_time        = sum(request_times) / len(request_times)
    remaining_n     = total - processed_total
    eta_sec         = avg_time * remaining_n

    if n % 10 == 0 or n == len(df_todo):
        print(
            f"  Processed {processed_total:,}/{total:,} tweets "
            f"({pct_done:.1f}% done) | "
            f"avg {avg_time:.1f}s/tweet | "
            f"ETA ~{eta_sec/60:.1f} min"
        )

    # ── Checkpoint every N tweets ─────────────────────────────────────────────
    if n % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(checkpoint_path, index=False)
        print(f"  [Checkpoint saved: {len(results):,} rows → {checkpoint_path.name}]")

# ── Done ─────────────────────────────────────────────────────────────────────
elapsed    = time.time() - start_time
df_results = pd.DataFrame(results)

null_count = df_results['pred_label'].isnull().sum()

# Save final raw results and remove checkpoint
df_results.to_csv(raw_path, index=False)
if checkpoint_path.exists():
    checkpoint_path.unlink()
    print(f"Checkpoint removed (full results saved).")

print(f"\nDone! {len(df_results):,} classified in {elapsed/60:.1f} min")
print(f"Null / parse failures: {null_count} ({null_count/len(df_results)*100:.1f}%)")
print(f"Raw results saved: {raw_path}")

# ── Parse method distribution ─────────────────────────────────────────────────
print(f"\n--- Parse Method Distribution ---")
parse_dist = df_results['parse_method'].value_counts()
for method, count in parse_dist.items():
    pct = count / len(df_results) * 100
    print(f"  {method:<20}: {count:,} ({pct:.1f}%)")

## 7. Handle Errors & Finalize Predictions

In [ ]:
# Show error rate
null_mask = df_results['pred_label'].isnull()
print(f"Null predictions: {null_mask.sum()} / {len(df_results)} ({null_mask.sum()/len(df_results)*100:.1f}%)")

if null_mask.sum() > 0:
    print("\nSample failed predictions:")
    print(df_results[null_mask][['text', 'reasoning', 'parse_method']].head(3).to_string())

# Strategy: fill null predictions with majority class (reliable)
# This is conservative — avoids inflating misinformation recall
majority_class = df_results['true_label'].mode()[0]
df_results['pred_label_final'] = df_results['pred_label'].fillna(majority_class)

# For empty/default parse_method rows that were filled, set confidence to 0.5
# so they don't pollute calibration analysis with confidence=0.0
empty_filled = df_results['pred_label'].isnull()
df_results.loc[empty_filled, 'confidence'] = df_results.loc[empty_filled, 'confidence'].replace(0.0, 0.5)

print(f"\nFilled {null_mask.sum()} nulls with majority class: '{majority_class}'")
print(f"\nPrediction distribution:")
print(df_results['pred_label_final'].value_counts())

print(f"\n--- Parse Method Summary ---")
parse_dist = df_results['parse_method'].value_counts()
for method, count in parse_dist.items():
    pct = count / len(df_results) * 100
    print(f"  {method:<20}: {count:,} ({pct:.1f}%)")

## 8. Evaluation Metrics

In [ ]:
y_true = df_results['true_label'].map(LABEL_MAP).values
y_pred = df_results['pred_label_final'].map(LABEL_MAP).values

print(f"\n{'='*60}")
print(f" {DATASET.upper()} - Ollama {OLLAMA_MODEL} Zero-Shot Results")
print(f"{'='*60}")
print(classification_report(y_true, y_pred, target_names=CFG['label_names'], digits=4))

pos_label_int = LABEL_MAP[CFG['pos_label']]

# Parse method counts (for summary CSV)
parse_counts = df_results['parse_method'].value_counts().to_dict()

metrics = {
    'dataset':         DATASET,
    'model':           f'ollama-zero-shot ({OLLAMA_MODEL})',
    'accuracy':        accuracy_score(y_true, y_pred),
    'f1_macro':        f1_score(y_true, y_pred, average='macro'),
    'f1_weighted':     f1_score(y_true, y_pred, average='weighted'),
    'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
    'recall_macro':    recall_score(y_true, y_pred, average='macro', zero_division=0),
    f'f1_{CFG["pos_label"]}': f1_score(y_true, y_pred, pos_label=pos_label_int, zero_division=0),
    'null_predictions':        int(null_mask.sum()),
    'parse_errors':            int(df_results['parse_error'].sum()),
    'parse_json':              parse_counts.get('json', 0),
    'parse_keyword_fallback':  parse_counts.get('keyword_fallback', 0),
    'parse_empty':             parse_counts.get('empty', 0),
    'parse_default':           parse_counts.get('default', 0),
}

print("--- Summary ---")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<35}: {v:.4f}")
    else:
        print(f"  {k:<35}: {v}")

## 9. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=CFG['label_names'],
            yticklabels=CFG['label_names'], ax=axes[0])
axes[0].set_title(f'{DATASET.upper()} - Zero-Shot Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Oranges',
            xticklabels=CFG['label_names'],
            yticklabels=CFG['label_names'], ax=axes[1])
axes[1].set_title(f'{DATASET.upper()} - Zero-Shot Confusion Matrix (Normalized)', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_zeroshot_cm.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 10. Confidence Distribution Analysis

In [ ]:
df_results['correct'] = (df_results['true_label'] == df_results['pred_label_final'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence by correctness
correct_conf   = df_results[df_results['correct']]['confidence']
incorrect_conf = df_results[~df_results['correct']]['confidence']

axes[0].hist(correct_conf,   bins=20, alpha=0.6, color='steelblue', label=f'Correct (n={len(correct_conf)})')
axes[0].hist(incorrect_conf, bins=20, alpha=0.6, color='crimson',   label=f'Incorrect (n={len(incorrect_conf)})')
axes[0].set_xlabel('Model Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title(f'{DATASET.upper()} - Confidence by Prediction Outcome', fontweight='bold')
axes[0].legend()
axes[0].axvline(0.5, color='black', linestyle='--', alpha=0.5)

# Confidence by predicted label
for label_name in CFG['label_names']:
    subset = df_results[df_results['pred_label_final'] == label_name]['confidence']
    axes[1].hist(subset, bins=20, alpha=0.6, label=f'{label_name} (n={len(subset)})')
axes[1].set_xlabel('Model Confidence')
axes[1].set_ylabel('Count')
axes[1].set_title(f'{DATASET.upper()} - Confidence by Predicted Label', fontweight='bold')
axes[1].legend()

plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_zeroshot_confidence.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean confidence (correct):   {correct_conf.mean():.3f}")
print(f"Mean confidence (incorrect): {incorrect_conf.mean():.3f}")

## 10b. Fallback Parse Analysis

In [ ]:
# ── Fallback parse analysis ───────────────────────────────────────────────────
# Shows how many predictions relied on fallback parsing vs clean JSON,
# their confidence distributions, and flags suspiciously uniform confidence.

print(f"{'='*60}")
print(f" Parse Method Analysis — {DATASET.upper()}")
print(f"{'='*60}")

parse_groups = df_results.groupby('parse_method')['confidence']

for method, group in parse_groups:
    n     = len(group)
    mean  = group.mean()
    std   = group.std()
    pct   = n / len(df_results) * 100
    print(f"\n  [{method}]  n={n:,} ({pct:.1f}%)")
    print(f"    mean confidence : {mean:.3f}  |  std : {std:.3f}")

    # Flag suspiciously uniform confidence (all at 0.5 ± 0.01)
    near_half = ((group - 0.5).abs() < 0.01).sum()
    if near_half == n and n > 0:
        print(f"    *** WARNING: ALL {n} predictions have confidence ≈ 0.5 — likely placeholder values ***")
    elif near_half / max(n, 1) > 0.8:
        print(f"    ** NOTICE: {near_half}/{n} predictions have confidence ≈ 0.5 ({near_half/n*100:.0f}%) **")

# Summary counts
fallback_total = df_results[df_results['parse_method'] != 'json'].shape[0]
json_total     = df_results[df_results['parse_method'] == 'json'].shape[0]
print(f"\n  JSON parsed        : {json_total:,} ({json_total/len(df_results)*100:.1f}%)")
print(f"  Fallback (all)     : {fallback_total:,} ({fallback_total/len(df_results)*100:.1f}%)")

# Visualise confidence distributions per parse method
parse_method_names = df_results['parse_method'].unique().tolist()
n_methods = len(parse_method_names)

fig, axes = plt.subplots(1, n_methods, figsize=(6 * n_methods, 4), sharey=False)
if n_methods == 1:
    axes = [axes]

colors = {'json': 'steelblue', 'keyword_fallback': 'darkorange', 'empty': 'crimson', 'default': 'grey'}
for ax, method in zip(axes, parse_method_names):
    subset = df_results[df_results['parse_method'] == method]['confidence']
    color  = colors.get(method, 'grey')
    ax.hist(subset, bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(0.5, color='black', linestyle='--', alpha=0.6, label='0.5 line')
    ax.set_title(f'{method}\n(n={len(subset):,})', fontweight='bold')
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle(f'{DATASET.upper()} — Confidence by Parse Method', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_zeroshot_parse_confidence.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSaved: {fig_path}")

## 11. Error Analysis — What Did Ollama Get Wrong?

In [ ]:
# Save final predictions (parse_method column now included)
df_results['true_label_int'] = df_results['true_label'].map(LABEL_MAP)
df_results['pred_label_int'] = df_results['pred_label_final'].map(LABEL_MAP)

pred_path = PREDS_DIR / f'{DATASET}_zeroshot_test_predictions.csv'
df_results.to_csv(pred_path, index=False)
print(f"Predictions saved: {pred_path}")
print(f"  Columns: {list(df_results.columns)}")

# Save summary (same filename as before so notebook 09 works unchanged)
# Now also includes parse_json / parse_keyword_fallback / parse_empty / parse_default counts
summary_path = PREDS_DIR / f'{DATASET}_zeroshot_summary.csv'
pd.DataFrame([metrics]).to_csv(summary_path, index=False)
print(f"Summary saved: {summary_path}")

print("\n=== FINAL ZERO-SHOT RESULTS ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<35}: {v:.4f}")
    else:
        print(f"  {k:<35}: {v}")

## 12. Save Final Predictions & Summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')

pos_key = f'f1_{CFG["pos_label"]}'
metrics_display = [
    ['Metric', 'Value'],
    ['Accuracy',               f"{metrics['accuracy']:.4f}"],
    ['F1 Macro',               f"{metrics['f1_macro']:.4f}"],
    ['F1 Weighted',            f"{metrics['f1_weighted']:.4f}"],
    ['Precision (Macro)',      f"{metrics['precision_macro']:.4f}"],
    ['Recall (Macro)',         f"{metrics['recall_macro']:.4f}"],
    [f'F1 ({CFG["pos_label"]})', f"{metrics[pos_key]:.4f}"],
    ['Parse Errors',           str(metrics['parse_errors'])],
    ['Parse: json',            str(metrics['parse_json'])],
    ['Parse: keyword_fallback',str(metrics['parse_keyword_fallback'])],
    ['Parse: empty',           str(metrics['parse_empty'])],
    ['Parse: default',         str(metrics['parse_default'])],
]

table = ax.table(
    cellText=metrics_display[1:],
    colLabels=metrics_display[0],
    cellLoc='center',
    loc='center',
    bbox=[0.15, 0, 0.7, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(10)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#e67e22')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#fef9f0')

ax.set_title(f'{DATASET.upper()} - Ollama {OLLAMA_MODEL} Zero-Shot Results', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_zeroshot_summary_table.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 13. Results Summary Table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

pos_key = f'f1_{CFG["pos_label"]}'
metrics_display = [
    ['Metric', 'Value'],
    ['Accuracy',           f"{metrics['accuracy']:.4f}"],
    ['F1 Macro',           f"{metrics['f1_macro']:.4f}"],
    ['F1 Weighted',        f"{metrics['f1_weighted']:.4f}"],
    ['Precision (Macro)',  f"{metrics['precision_macro']:.4f}"],
    ['Recall (Macro)',     f"{metrics['recall_macro']:.4f}"],
    [f'F1 ({CFG["pos_label"]})', f"{metrics[pos_key]:.4f}"],
    ['Parse Errors',       str(metrics['parse_errors'])],
]

table = ax.table(
    cellText=metrics_display[1:],
    colLabels=metrics_display[0],
    cellLoc='center',
    loc='center',
    bbox=[0.2, 0, 0.6, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(11)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#e67e22')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#fef9f0')

ax.set_title(f'{DATASET.upper()} - Ollama {OLLAMA_MODEL} Zero-Shot Results', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_zeroshot_summary_table.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")